In [38]:
import os
import pandas as pd
from tqdm import tqdm
from datetime import datetime, timedelta
import numpy as np
import matplotlib.pyplot as plt
import math
from collections import defaultdict
import scienceplots
plt.style.use('science')

save_root_dir = 'archive/tennis_abstract_dfs_good_Elos'
backup_root_dir = 'archive/tennis_abstract_dfs_Elo_backup'
big_df_name = 'big_df.pkl'
df_root_dir = 'tennis_abstract_dfs'
html_root_dir = 'tennis_abstract'
extended_df_root_dir = 'tennis_abstract_dfs_extended'
extended_195_df_root_dir = 'tennis_abstract_dfs_extended_195'
surfs = ['Clay', 'Grass', 'Hard']
# poly3d_coeefs = [-3.62592888,  5.42864825, -0.86041857,  0.02900003]
poly3d_coeefs = [-3.51591704,  5.26301386, -0.79325641,  0.02302555]
swr_baseline = 63.5
rwr_baseline = 36.5


In [39]:
round_vals = {
    'F': 1,
    'SF': 2,
    'QF': 3,
    'R16': 4,
    'R32': 5,
    'R64': 6,
    'R128': 7,
    'Q3': 8,
    'Q2': 9,
    'Q1': 10
}
round_vals = defaultdict(lambda: 11, round_vals)


def calc_serve_win_rate(row):
    elems = ['first_serves_won', 'second_serves_won', 'service_pts']
    if any([row[elem] == '' for elem in elems]):
        return None
    denom = float(row['service_pts'])
    if denom == 0:
        return 0
    return (float(row['first_serves_won']) + float(row['second_serves_won'])) / denom

def calc_overall_win_rate(row):
    elems = ['first_serves_won', 'second_serves_won', 'ofirst_serves_won', 'osecond_serves_won', 'service_pts', 'oservice_pts']
    if any([row[elem] == '' for elem in elems]):
        return None
    denom = float(row['service_pts']) + float(row['oservice_pts'])
    if denom == 0:
        return 0
    player_points_won = float(row['first_serves_won']) + float(row['second_serves_won'])
    opp_points_won = float(row['oservice_pts']) - float(row['ofirst_serves_won']) - float(row['osecond_serves_won'])
    return (player_points_won + opp_points_won) / denom

def calc_return_win_rate(row):
    elems = ['ofirst_serves_won', 'osecond_serves_won', 'oservice_pts']
    if any([row[elem] == '' for elem in elems]):
        return None
    denom = float(row['oservice_pts'])
    if denom == 0:
        return 0
    return 1 - ((float(row['ofirst_serves_won']) + float(row['osecond_serves_won'])) / denom)

def calc_first_serve_win_rate(row):
    elems = ['first_serves_won', 'first_serves_in']
    if any([row[elem] == '' for elem in elems]):
        return None
    denom = float(row['first_serves_in'])
    if denom == 0:
        return 0
    return float(row['first_serves_won']) / denom

def calc_first_serve_in_rate(row):
    elems = ['service_pts', 'first_serves_in']
    if any([row[elem] == '' for elem in elems]):
        return None
    denom = float(row['service_pts'])
    if denom == 0:
        return 0
    return float(row['first_serves_in']) / denom

def calc_first_return_in_rate(row):
    elems = ['oservice_pts', 'ofirst_serves_in']
    if any([row[elem] == '' for elem in elems]):
        return None
    denom = float(row['oservice_pts'])
    if denom == 0:
        return 0
    return float(row['ofirst_serves_in']) / denom

def calc_second_serve_win_rate(row):
    elems = ['second_serves_won', 'service_pts', 'first_serves_in']
    if any([row[elem] == '' for elem in elems]):
        return None
    denom = (float(row['service_pts']) - float(row['first_serves_in']))
    if denom == 0:
        return 0
    return float(row['second_serves_won']) / denom
def calc_first_return_win_rate(row):
    elems = ['ofirst_serves_won', 'ofirst_serves_in']
    if any([row[elem] == '' for elem in elems]):
        return None
    denom = float(row['ofirst_serves_in'])
    if denom == 0:
        return 0
    return 1 - (float(row['ofirst_serves_won']) / denom)

def calc_second_return_win_rate(row):
    elems = ['osecond_serves_won', 'oservice_pts', 'ofirst_serves_in']
    if any([row[elem] == '' for elem in elems]):
        return None
    denom = (float(row['oservice_pts']) - float(row['ofirst_serves_in']))
    if denom == 0:
        return 0
    return 1-(float(row['osecond_serves_won']) / denom)

In [40]:
def analytic_set_prob_points(ps, pr):
 # --- Define common sub-expressions for readability and accuracy ---
    pr_minus_1 = pr - 1
    ps_minus_1 = ps - 1

    # pr-related common terms
    pr_term_1plus2pr = (1 + 2 * pr)
    pr_term_5plus4pr_minus2pr = (5 + 4 * (-2 + pr) * pr)  # 5 - 8pr + 4pr^2
    pr_term_1plus2pr_minus1pr = (1 + 2 * pr_minus_1 * pr)  # 1 - 2pr + 2pr^2
    pr_term_1plus4pr_sq = (1 + 4 * pr**2)
    pr_term_minus3plus2pr = (-3 + 2 * pr)
    pr_term_3minus2pr = (3 - 2 * pr)

    # ps-related common terms
    ps_term_1plus2ps = (1 + 2 * ps)
    ps_term_5plus4ps_minus2ps = (5 + 4 * (-2 + ps) * ps)  # 5 - 8ps + 4ps^2
    ps_term_1plus2ps_minus1ps = (1 + 2 * ps_minus_1 * ps)  # 1 - 2ps + 2ps^2
    ps_term_1plus4ps_sq = (1 + 4 * ps**2)
    ps_term_minus3plus2ps = (-3 + 2 * ps)
    ps_term_3minus2ps = (3 - 2 * ps)

    # --- Calculate the first 15 main terms of the numerator ---

    term1 = (
        4 * (pr_minus_1)**12 * pr**4 * pr_term_minus3plus2pr * pr_term_1plus2pr**3 *
        pr_term_5plus4pr_minus2pr * pr_term_1plus2pr_minus1pr**2 * pr_term_1plus4pr_sq**3 *
        ps**20 * ps_term_minus3plus2ps**5 * ps_term_5plus4ps_minus2ps**5 *
        ps_term_1plus2ps_minus1ps
    )

    term2 = (
        pr_minus_1**16 * pr**4 * pr_term_minus3plus2pr * pr_term_1plus2pr**4 *
        pr_term_5plus4pr_minus2pr * pr_term_1plus2pr_minus1pr * pr_term_1plus4pr_sq**4 *
        ps**20 * ps_term_minus3plus2ps**5 * ps_term_5plus4ps_minus2ps**5 *
        ps_term_1plus2ps_minus1ps
    )

    term3 = (
        3 * pr_term_3minus2pr**2 * pr_minus_1**4 * pr**8 * pr_term_1plus2pr *
        pr_term_5plus4pr_minus2pr**2 * pr_term_1plus2pr_minus1pr**3 * pr_term_1plus4pr_sq *
        ps_term_3minus2ps**4 * ps**16 * ps_term_5plus4ps_minus2ps**4 *
        ps_term_1plus2ps_minus1ps**2
    )

    term4 = (
        3 * pr_term_3minus2pr**2 * pr_minus_1**8 * pr**8 * pr_term_1plus2pr**2 *
        pr_term_5plus4pr_minus2pr**2 * pr_term_1plus2pr_minus1pr**2 * pr_term_1plus4pr_sq**2 *
        ps_term_3minus2ps**4 * ps**16 * ps_term_5plus4ps_minus2ps**4 *
        ps_term_1plus2ps_minus1ps**2
    )

    term5 = (
        pr**12 * pr_term_minus3plus2pr**3 * pr_term_5plus4pr_minus2pr**3 *
        pr_term_1plus2pr_minus1pr**3 * ps**12 * ps_term_minus3plus2ps**3 *
        ps_term_5plus4ps_minus2ps**3 * ps_term_1plus2ps_minus1ps**3
    )

    term6 = (
        24 * pr_term_3minus2pr**2 * pr_minus_1**8 * pr**8 * pr_term_1plus2pr**2 *
        pr_term_5plus4pr_minus2pr**2 * pr_term_1plus2pr_minus1pr**2 * pr_term_1plus4pr_sq**2 *
        ps_term_3minus2ps**4 * ps_minus_1**4 * ps**16 * ps_term_1plus2ps *
        ps_term_5plus4ps_minus2ps**4 * ps_term_1plus2ps_minus1ps * ps_term_1plus4ps_sq
    )

    term7 = (
        20 * pr_term_3minus2pr**2 * pr_minus_1**12 * pr**8 * pr_term_1plus2pr**3 *
        pr_term_5plus4pr_minus2pr**2 * pr_term_1plus2pr_minus1pr * pr_term_1plus4pr_sq**3 *
        ps_term_3minus2ps**4 * ps_minus_1**4 * ps**16 * ps_term_1plus2ps *
        ps_term_5plus4ps_minus2ps**4 * ps_term_1plus2ps_minus1ps * ps_term_1plus4ps_sq
    )

    term8 = (
        3 * pr**12 * pr_term_minus3plus2pr**3 * pr_term_5plus4pr_minus2pr**3 *
        pr_term_1plus2pr_minus1pr**3 * ps_minus_1**4 * ps**12 *
        ps_term_minus3plus2ps**3 * ps_term_1plus2ps * ps_term_5plus4ps_minus2ps**3 *
        ps_term_1plus2ps_minus1ps**2 * ps_term_1plus4ps_sq
    )

    term9 = (
        12 * pr_minus_1**4 * pr**12 * pr_term_minus3plus2pr**3 * pr_term_1plus2pr *
        pr_term_5plus4pr_minus2pr**3 * pr_term_1plus2pr_minus1pr**2 * pr_term_1plus4pr_sq *
        ps_minus_1**4 * ps**12 * ps_term_minus3plus2ps**3 * ps_term_1plus2ps *
        ps_term_5plus4ps_minus2ps**3 * ps_term_1plus2ps_minus1ps**2 * ps_term_1plus4ps_sq
    )

    term10 = (
        24 * pr_minus_1**4 * pr**12 * pr_term_minus3plus2pr**3 * pr_term_1plus2pr *
        pr_term_5plus4pr_minus2pr**3 * pr_term_1plus2pr_minus1pr**2 * pr_term_1plus4pr_sq *
        ps_minus_1**8 * ps**12 * ps_term_minus3plus2ps**3 * ps_term_1plus2ps**2 *
        ps_term_5plus4ps_minus2ps**3 * ps_term_1plus2ps_minus1ps * ps_term_1plus4ps_sq**2
    )

    term11 = (
        60 * pr_minus_1**8 * pr**12 * pr_term_minus3plus2pr**3 * pr_term_1plus2pr**2 *
        pr_term_5plus4pr_minus2pr**3 * pr_term_1plus2pr_minus1pr * pr_term_1plus4pr_sq**2 *
        ps_minus_1**8 * ps**12 * ps_term_minus3plus2ps**3 * ps_term_1plus2ps**2 *
        ps_term_5plus4ps_minus2ps**3 * ps_term_1plus2ps_minus1ps * ps_term_1plus4ps_sq**2
    )

    term12 = (
        6 * pr_term_3minus2pr**4 * pr**16 * pr_term_5plus4pr_minus2pr**4 *
        pr_term_1plus2pr_minus1pr**2 * ps_term_3minus2ps**2 * ps_minus_1**8 *
        ps**8 * ps_term_1plus2ps**2 * ps_term_5plus4ps_minus2ps**2 *
        ps_term_1plus2ps_minus1ps**2 * ps_term_1plus4ps_sq**2
    )

    term13 = (
        4 * pr_term_3minus2pr**4 * pr**16 * pr_term_5plus4pr_minus2pr**4 *
        pr_term_1plus2pr_minus1pr**2 * ps_term_3minus2ps**2 * ps_minus_1**12 *
        ps**8 * ps_term_1plus2ps**3 * ps_term_5plus4ps_minus2ps**2 *
        ps_term_1plus2ps_minus1ps * ps_term_1plus4ps_sq**3
    )

    term14 = (
        40 * pr_term_3minus2pr**4 * pr_minus_1**4 * pr**16 * pr_term_1plus2pr *
        pr_term_5plus4pr_minus2pr**4 * pr_term_1plus2pr_minus1pr * pr_term_1plus4pr_sq *
        ps_term_3minus2ps**2 * ps_minus_1**12 * ps**8 * ps_term_1plus2ps**3 *
        ps_term_5plus4ps_minus2ps**2 * ps_term_1plus2ps_minus1ps * ps_term_1plus4ps_sq**3
    )

    term15 = (
        5 * pr**20 * pr_term_minus3plus2pr**5 * pr_term_5plus4pr_minus2pr**5 *
        pr_term_1plus2pr_minus1pr * ps_minus_1**16 * ps**4 * ps_term_minus3plus2ps *
        ps_term_1plus2ps**4 * ps_term_5plus4ps_minus2ps * ps_term_1plus2ps_minus1ps *
        ps_term_1plus4ps_sq**4
    )

    sum_of_first_15_terms = (
        term1 + term2 + term3 + term4 + term5 + term6 + term7 + term8 +
        term9 + term10 + term11 + term12 + term13 + term14 + term15
    )

    # --- Calculate the complex fractional term ---

    # Denominator of the fraction part: (1 - ps + pr (-1 + 2 ps))
    frac_den_main = (1 - ps + pr * (-1 + 2 * ps))
    if frac_den_main == 0:
        raise ValueError("Division by zero: Denominator of the inner fraction is zero.")

    # Numerator Block 1 (the first large parenthesized block after 1/denominator)
    num_block1_term1 = (
        -((pr_minus_1)**20 * pr_term_1plus2pr**5 * pr_term_1plus4pr_sq**5 *
        ps**20 * ps_term_minus3plus2ps**5 * ps_term_5plus4ps_minus2ps**5)
    )

    num_block1_term2 = (
        -25 * (pr_minus_1)**16 * pr**4 * pr_term_minus3plus2pr * pr_term_1plus2pr**4 *
        pr_term_5plus4pr_minus2pr * pr_term_1plus4pr_sq**4 * ps_term_3minus2ps**4 *
        (ps_minus_1)**4 * ps**16 * ps_term_1plus2ps * ps_term_5plus4ps_minus2ps**4 *
        ps_term_1plus4ps_sq
    )

    num_block1_term3 = (
        -100 * pr_term_3minus2pr**2 * (pr_minus_1)**12 * pr**8 * pr_term_1plus2pr**3 *
        pr_term_5plus4pr_minus2pr**2 * pr_term_1plus4pr_sq**3 * (ps_minus_1)**8 *
        ps**12 * ps_term_minus3plus2ps**3 * ps_term_1plus2ps**2 *
        ps_term_5plus4ps_minus2ps**3 * ps_term_1plus4ps_sq**2
    )

    num_block1_term4 = (
        -100 * (pr_minus_1)**8 * pr**12 * pr_term_minus3plus2pr**3 * pr_term_1plus2pr**2 *
        pr_term_5plus4pr_minus2pr**3 * pr_term_1plus4pr_sq**2 * ps_term_3minus2ps**2 *
        (ps_minus_1)**12 * ps**8 * ps_term_1plus2ps**3 * ps_term_5plus4ps_minus2ps**2 *
        ps_term_1plus4ps_sq**3
    )

    num_block1_term5 = (
        -25 * pr_term_3minus2pr**4 * (pr_minus_1)**4 * pr**16 * pr_term_1plus2pr *
        pr_term_5plus4pr_minus2pr**4 * pr_term_1plus4pr_sq * (ps_minus_1)**16 *
        ps**4 * ps_term_minus3plus2ps * ps_term_1plus2ps**4 * ps_term_5plus4ps_minus2ps *
        ps_term_1plus4ps_sq**4
    )

    num_block1_term6 = (
        -pr**20 * pr_term_minus3plus2pr**5 * pr_term_5plus4pr_minus2pr**5 *
        (ps_minus_1)**20 * ps_term_1plus2ps**5 * ps_term_1plus4ps_sq**5
    )

    numerator_block1_sum = (
        num_block1_term1 + num_block1_term2 + num_block1_term3 +
        num_block1_term4 + num_block1_term5 + num_block1_term6
    )

    # Numerator Block 2 (the second large parenthesized block after 1/denominator)

    num_block2_term1 = (
        pr**4 * pr_term_minus3plus2pr * pr_term_5plus4pr_minus2pr *
        ps**4 * ps_term_minus3plus2ps * ps_term_5plus4ps_minus2ps *
        (1 - ps + pr * (-1 + 2 * ps))
    )

    # Inner part of num_block2_term2 (parenthesis before the long polynomial)
    num_block2_term2_inner_parens = (
        -((pr_minus_1)**4 * pr_term_1plus2pr * pr_term_1plus4pr_sq *
        ps**4 * ps_term_minus3plus2ps * ps_term_5plus4ps_minus2ps) -
        (pr**4 * pr_term_minus3plus2pr * pr_term_5plus4pr_minus2pr *
        (ps_minus_1)**4 * ps_term_1plus2ps * ps_term_1plus4ps_sq)
    )

    # Long polynomial part of num_block2_term2
    poly_term1 = (6 - 5 * ps) * ps**5
    poly_term2 = 15 * pr * ps_minus_1 * ps**4 * (-6 + 5 * ps)
    poly_term3 = -5 * pr**2 * ps_minus_1 * ps**3 * (60 + ps * (-141 + 70 * ps))
    poly_term4 = (
        5 * pr**6 * ps_minus_1 * (1 + 14 * ps_minus_1 * ps * (1 + 3 * ps_minus_1 * ps))
    )
    poly_term5 = (
        5 * pr**3 * ps_minus_1 * ps**2 * (-60 + ps * (295 + 28 * ps * (-14 + 5 * ps)))
    )
    poly_term6 = (
        -3 * pr**4 * ps_minus_1 * ps * (30 + ps * (-305 + 42 * ps * (20 + ps * (-19 + 5 * ps))))
    )
    poly_term7 = (
        pr**5 * ps_minus_1 * (-6 + ps * (159 + 14 * ps * (-64 + 3 * ps * (42 + 5 * (-6 + ps) * ps))))
    )

    long_polynomial_sum = (
        poly_term1 + poly_term2 + poly_term3 + poly_term4 +
        poly_term5 + poly_term6 + poly_term7
    )

    num_block2_term2 = (
        pr * ps * num_block2_term2_inner_parens * long_polynomial_sum
    )

    numerator_block2_sum = num_block2_term1 + num_block2_term2

    # Combine the fractional term
    fractional_term_value = (numerator_block1_sum * numerator_block2_sum) / frac_den_main

    # --- Calculate the overall result ---
    total_numerator = sum_of_first_15_terms + fractional_term_value

    # Overall denominator for the entire expression
    overall_denominator = pr_term_1plus2pr_minus1pr**6 * ps_term_1plus2ps_minus1ps**6
    if overall_denominator == 0:
        raise ValueError("Division by zero: Overall denominator is zero.")

    final_result = total_numerator / overall_denominator

    return final_result

In [41]:
def analytic_match_prob(ps, pr, sets = 3):
    p = analytic_set_prob_points(ps, pr)
    if sets == 3:
        prob = p**2 + 2 * (p**2)*(1-p) 
    elif sets == 5:
        prob = p**3 + 3 * (p**3)*(1-p) + 6 * (p**3)*(1-p)**2
    else:
        raise ValueError(f"Unsupported number of sets: {sets}. Only 3 or 5 sets are supported.")
    return prob

In [42]:
def restore_df():
    backup_path = f'{backup_root_dir}/{big_df_name}'
    file_path = f'{big_df_name}'
    og = pd.read_pickle(backup_path)
    pd.to_pickle(og, file_path)


def save_df():
    save_path = f'{save_root_dir}/{big_df_name}'
    file_path = f'{big_df_name}'
    og = pd.read_pickle(file_path)
    pd.to_pickle(og, save_path)

def backup_df():
    save_path = f'{backup_root_dir}/{big_df_name}'
    file_path = f'{big_df_name}'
    og = pd.read_pickle(file_path)
    pd.to_pickle(og, save_path)

# save_df()
# backup_df()
# restore_df()

big_df = pd.read_pickle(big_df_name)

all_names = set(big_df['formatted_player']).union(big_df['formatted_opp'])
all_names = list(all_names)
len(all_names)


195

In [ ]:
for name in all_names:
    extended_df = pd.read_pickle(f'{extended_df_root_dir}/{name}.pkl')
    extended_df['round_val'] = extended_df['round'].apply(lambda x: round_vals[x])
    extended_df['serve_win_rate'] = extended_df.apply(calc_serve_win_rate, axis=1)
    extended_df['overall_win_rate'] = extended_df.apply(calc_overall_win_rate, axis=1)
    extended_df['return_win_rate'] = extended_df.apply(calc_return_win_rate, axis=1)
    extended_df['first_serve_win_rate'] = extended_df.apply(calc_first_serve_win_rate, axis=1)
    extended_df['first_serve_in_rate'] = extended_df.apply(calc_first_serve_in_rate, axis=1)
    extended_df['first_return_in_rate'] = extended_df.apply(calc_first_return_in_rate, axis=1)
    extended_df['second_serve_win_rate'] = extended_df.apply(calc_second_serve_win_rate, axis=1)
    extended_df['first_return_win_rate'] = extended_df.apply(calc_first_return_win_rate, axis=1)
    extended_df['second_return_win_rate'] = extended_df.apply(calc_second_return_win_rate, axis=1)
    extended_df['p_matchnum'] = extended_df.index + 1
    extended_df['result'] = extended_df['win/loss'].apply(lambda x: 1 if x == 'W' else 0)
    extended_df['ranking_superiority'] = extended_df['rank'] > extended_df['orank']
    def safe_post_win_prob(row):
        if pd.isnull(row['serve_win_rate']) or pd.isnull(row['return_win_rate']):
            return np.nan
        sets = int(row['max_num_of_sets']) if 'max_num_of_sets' in row else 3
        return analytic_match_prob(row['serve_win_rate'], row['return_win_rate'], sets=sets)

    extended_df['post_win_prob'] = extended_df.apply(safe_post_win_prob, axis=1)
    extended_df['formatted_opp'] = extended_df['opp'].apply(lambda x: x.replace(' ', ''))
    extended_df.to_pickle(f'{extended_df_root_dir}/{name}.pkl')

In [ ]:
alcaraz_df = pd.read_pickle(f'{extended_df_root_dir}/CarlosAlcaraz.pkl')

0.9994839015935083

In [45]:
for name in all_names:
    extended_df = pd.read_pickle(f'{extended_df_root_dir}/{name}.pkl')
    pd.to_pickle(extended_df, f'{extended_195_df_root_dir}/{name}.pkl')
    og_df = pd.read_pickle(f'{df_root_dir}/{name}.pkl')
    diff = set(og_df.columns) - set(extended_df.columns)
    filter_substrings = ['R_', 'prev', 'E_', 'elo_', 'sr']
    filter_strings = ['win_prob', 'ovrl_ovrl_win_prob', 'ovrl_err', 'ovrl_mxed_win_prob', 'swr_err', 'rwr_err', 'o_matchnum', 'ovrl_surf_win_prob']
    filtered = [col for col in diff if (not any(sub in col for sub in filter_substrings) and not any(sub == col for sub in filter_strings))]
    print(filtered)
    break

['formatted_opp', 'ranking_correct']
